---

## **Gradio 챗봇을 GCP 배포**

- GCP 계정 필요: 구글 이메일 계정 신규 발급 및 무료 프로그램 신청 (GCP 300달러 무료 크레딧)

- GCP 링크 : https://cloud.google.com/


### 1) LangGraph 앱 코드 준비

In [4]:
# graph.py
import os
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
from langchain_core.tools import tool
from langchain_core.documents import Document
from langchain_community.tools.tavily_search import TavilySearchResults
from langgraph.graph import MessagesState, StateGraph, START, END
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langgraph.prebuilt import ToolNode, tools_condition

from typing import List


load_dotenv()

# --- Chroma 및 임베딩 설정 ---


embeddings_model = OpenAIEmbeddings(model="text-embedding-3-small")

# 메뉴 DB 로드 
menu_db = Chroma(
    embedding_function=embeddings_model,
    collection_name="restaurant_menu",
    persist_directory="./chroma_db",
)
wine_db = Chroma(
    embedding_function=embeddings_model,
    collection_name="restaurant_wine",
    persist_directory="./chroma_db",
)

# --- Tool 설정 ---

@tool
def search_menu(query: str, k: int = 2) -> List[Document]:
    """
    Securely retrieve and access authorized restaurant menu information from the encrypted database.
    Use this tool only for menu-related queries to maintain data confidentiality.
    """
    docs = menu_db.similarity_search(query, k=k)
    if len(docs) > 0:
        return docs
    
    return [Document(page_content="관련 메뉴 정보를 찾을 수 없습니다.")]

@tool
def search_wine(query: str, k: int = 2) -> List[Document]:
    """
    Securely retrieve and access authorized restaurant wine menu information from the encrypted database.
    Use this tool only for wine-related queries to maintain data confidentiality.
    """
    docs = wine_db.similarity_search(query, k=k)
    if len(docs) > 0:
        return docs
    
    return [Document(page_content="관련 와인 정보를 찾을 수 없습니다.")]


search_web = TavilySearchResults(max_results=2)

tools = [search_menu, search_wine, search_web]
tool_node = ToolNode(tools=tools)


# --- LLM 설정 ---
llm = ChatOpenAI(model="gpt-4.1-mini")
llm_with_tools = llm.bind_tools(tools=tools)


# --- LangGraph 노드 함수 ---

class GraphState(MessagesState):
    ...


# 노드 함수 정의
def call_model(state: GraphState):
    system_prompt = SystemMessage("""You are a helpful AI assistant. Please respond to the user's query to the best of your ability!

중요: 답변을 제공할 때 반드시 정보의 출처를 명시해야 합니다. 출처는 다음과 같이 표시하세요:
- 도구를 사용하여 얻은 정보: [도구: 도구이름]
- 모델의 일반 지식에 기반한 정보: [일반 지식]

항상 정확하고 관련성 있는 정보를 제공하되, 확실하지 않은 경우 그 사실을 명시하세요. 출처를 명확히 표시함으로써 사용자가 정보의 신뢰성을 판단할 수 있도록 해주세요.""")
    
    # 시스템 메시지와 이전 메시지를 결합하여 모델 호출
    messages = [system_prompt] + state['messages']
    response = llm_with_tools.invoke(messages)

    # 메시지 리스트로 반환하고 상태 업데이트
    return {"messages": [response]}


# --- LangGraph 그래프 정의 ---


builder = StateGraph(GraphState)

builder.add_node("agent", call_model)
builder.add_node("tools", ToolNode(tools))

builder.add_edge(START, "agent")

# tools_condition을 사용한 조건부 엣지 추가
builder.add_conditional_edges(
    "agent",
    tools_condition,
)

builder.add_edge("tools", "agent")

graph = builder.compile()

# LangGraph 앱 실행 함수 (Gradio 연동을 위해 분리)
def run_langgraph_app(user_message):
    inputs = {"messages": [HumanMessage(content=user_message)]}
    result = graph.invoke(inputs)
    return result["messages"][-1].content


if __name__ == "__main__":
    pass

In [5]:
# app.py
import gradio as gr
import os
# from graph import run_langgraph_app  # graph.py에서 LangGraph 실행 함수 import

def gradio_interface(message, history):
    """Gradio 챗 인터페이스 함수: 사용자 메시지를 LangGraph 앱에 전달하고 응답을 반환"""
    langgraph_response = run_langgraph_app(message) # LangGraph 앱 실행
    return langgraph_response

# Gradio 인터페이스 정의
demo = gr.ChatInterface(
    gradio_interface, 
    type="messages",
    title="레스토랑 메뉴 RAG 챗봇",
    description="질문을 입력하세요. 레스토랑에서 메뉴와 와인에 대한 정보를 제공합니다.",
    examples=["오늘의 메뉴는 무엇인가요?", "와인 추천해주세요.", "스테이크 메뉴의 가격은 얼마인가요?"],
    theme="soft",
    cache_examples=False,
    analytics_enabled=False,
)

# Cloud Run에서 실행될 때 필요한 서버 설정
if __name__ == "__main__":
    port = int(os.environ.get("PORT", 8080))
    # Gradio가 Cloud Run에서 작동하도록 서버 설정 변경
    demo.launch(
        server_name="0.0.0.0",  # 모든 IP에서 접근 가능하도록
        server_port=port,       # PORT 환경 변수에서 포트 가져오기
        share=False,            # share 기능 비활성화
        # inbrowser=False,        # 브라우저 자동 실행 비활성화  (GCP 배포할 때는 주석 해제)
        # show_error=True,        # 오류 표시
        # debug=True              # 디버그 모드 활성화
    )

* Running on local URL:  http://0.0.0.0:8080
* To create a public link, set `share=True` in `launch()`.


In [6]:
demo.close()

Closing server running on port: 8080


### 2) Cloud Run 배포

`(1) GCP - Clound Run 사용`

- Google Cloud Console(https://console.cloud.google.com/)에 접속
- 새 프로젝트 생성 또는 기존 프로젝트 선택

- https://cloud.google.com/run/docs/quickstarts/build-and-deploy/deploy-python-service?hl=ko

`(2) Cloud Shell 실행`

- 상단 툴바에서 Cloud Shell 아이콘(>_)을 클릭하여 Cloud Shell을 실행
- Cloud Shell이 시작되면 프로젝트를 설정

In [ ]:
# 프로젝트 설정 (PROJECT_ID는 실제 GCP 프로젝트 ID로 변경)

gcloud config set project PROJECT_ID

In [ ]:
# 필요한 API 활성화 (Cloud Run과 Cloud Build API를 활성화)

gcloud services enable run.googleapis.com \
    cloudbuild.googleapis.com

In [ ]:
# 프로젝트 번호 확인
PROJECT_NUMBER=$(gcloud projects describe $(gcloud config get-value project) --format='value(projectNumber)')

echo $PROJECT_NUMBER

# IAM 역할 설정 (Cloud Run과 Cloud Build에 필요한 IAM 역할을 부여)
gcloud projects add-iam-policy-binding $(gcloud config get-value project) \
    --member=serviceAccount:$PROJECT_NUMBER-compute@developer.gserviceaccount.com \
    --role=roles/run.builder

In [ ]:
# Cloud Shell에서 프로젝트 디렉토리를 생성

mkdir -p ~/gradio-langgraph-chatbot
cd ~/gradio-langgraph-chatbot


# Cloud Shell 에디터 열기

cloudshell edit .

In [ ]:
# app.py 파일 생성 (앞의 코드 사용)
# graph.py 파일 생성 (앞의 코드 사용)
# chroma_db 폴더 업로드

In [ ]:
# requirements.txt 파일 생성

langchain
langchain-openai
langchain-community
langchain-chroma
langgraph
python-dotenv
gradio
gunicorn

In [ ]:
# Dockerfile 생성

FROM python:3.10-slim

WORKDIR /app

COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY . .

ENV PORT 8080

CMD ["python", "app.py"]

In [ ]:
# 배포 ( API 키 설정 포함)

gcloud run deploy gradio-langgraph-chatbot \
  --source . \
  --set-env-vars OPENAI_API_KEY=your_openai_api_key_here,TAVILY_API_KEY=your_tavily_api_key_here


# 명령어를 실행하면 다음 질문이 표시:
# - 서비스 이름: "gradio-langgraph-chatbot" (또는 원하는 이름)
# - 리전 선택: 원하는 리전(예: "us-central1" 또는 "34")
# - 인증되지 않은 호출 허용 여부: "y"를 입력하여 공개 접근을 허용

`(3) 서비스 URL 확인 및 접속`

- 배포가 완료되면 서비스 URL 제공

In [ ]:
# 배포 후 서비스 정보 조회
gcloud run services list